In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

In [ ]:
df = pd.read_csv("data.csv")
df.head()

In [ ]:
df[:1]

#EDA

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

## Absolute Frequency of Classes

In [ ]:
ax = sns.countplot(data=df, x='Bankrupt?')
ax.bar_label(ax.containers[0])

# Map ticks to custom labels
ax.set_xticks([0, 1])
ax.set_xticklabels(['0 (Not Bankrupt)', '1 (Bankrupt)'])

plt.xlabel("Bankrupt classes")
plt.ylabel("Frequency")
plt.title("Class balance");

## Distributions of "Net Income to Total Assets" column for both Classes

In [ ]:
sns.boxenplot(x="Bankrupt?" , y=" Net Income to Total Assets" , data=df)
plt.xlabel("Bankrupt classes")
plt.ylabel("Net Income to Total Assets")
plt.title("Distribution of Profit/ Net Income Ratio, by Class");

## Plotting Boxplots of the numerical features

In [ ]:
plt.figure(figsize = (20,20))
ax =sns.boxplot(data = df, orient="h")
ax.set_title('Bank Data Boxplots', fontsize = 18)
ax.set(xscale="log")
plt.show()

## Check for Multicollinearity with Correlation Heatmap (Spearman)

In [ ]:
f, ax = plt.subplots(figsize=(30, 25))
mat = df.corr('spearman')
mask = np.triu(np.ones_like(mat, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(mat, mask=mask, cmap=cmap, vmax=1, center=0,# annot = True,
            square=True, linewidths=.5, cbar_kws={"shrink": .5})
plt.show()

# Preprocessing

## Train Test Split

In [ ]:
x = df.drop(columns=['Bankrupt?'])
y = df['Bankrupt?']

x.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
print("X_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

## Get Accuracy of Baseline Model

In [ ]:
acc_baseline = y_train.value_counts(normalize=True).max()
print("Baseline Accuracy:", round(acc_baseline, 4))

# Hyperparameter Tuning with Random Forest
###Reasons to pick Random Forest
1. The dataset has 95 feature columns. Random Forest handles High Dimensionality well.
2. Random Forest is immmune to performance drops from Multicollinearity.
3. It is not affected by Outliers.
4. It does not need Feature Scaling.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV


rfc = RandomForestClassifier(class_weight='balanced', random_state=42)

In [ ]:
params = {
    "n_estimators": [125, 175],
    "max_depth": [2, 3, 5],
    "min_samples_split": [225, 275, 325],
    "min_samples_leaf": [90, 110, 130]
    }

In [ ]:
#Since minimizing False Negative(when Bankruptcy is predicted Not Bankruptcy) is more important in this case, we use 'recall'.
model = GridSearchCV(
    rfc,
    param_grid =params,
    cv=5,
    scoring='recall',  # Switches optimization focus from Accuracy to recall.
    n_jobs=-1,
    verbose=1)

In [ ]:
model.fit(x_train, y_train)

In [ ]:
pd.DataFrame(model.cv_results_).sort_values('rank_test_score').head()

In [ ]:
model.best_params_

#Evaluate

In [ ]:
train_recall = model.score(x_train , y_train)
test_recall = model.score(x_test , y_test)

#call .score() directly on best_estimator_, which defaults to accuracy
train_accuracy = model.best_estimator_.score(x_train, y_train)
test_accuracy = model.best_estimator_.score(x_test, y_test)

print(f"Training accuracy: {round(train_accuracy, 4)}")
print(f"Test accuracy: {round(test_accuracy, 4)}")
print(f"Training recall: {round(train_recall  , 4)}")
print(f"test recall: {round(test_recall  , 4)}")

## Classification Report and Confusion Matrix with predicted and actual values

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay , classification_report ,f1_score

In [ ]:
ConfusionMatrixDisplay.from_estimator(model,
                                      x_test,
                                      y_test);

In [ ]:
print(classification_report(y_test,
    model.predict(x_test)))

## Report with threshold

In [ ]:
# Get predicted probabilities for class 1 (Bankrupt)
y_probs = model.predict_proba(x_test)[:, 1]

custom_threshold = 0.25
y_pred_custom = (y_probs >= custom_threshold).astype(int)

print(f"--- Results with Threshold = {custom_threshold} ---")
print("Accuracy:", round(accuracy_score(y_test, y_pred_custom), 4))
print(classification_report(y_test, y_pred_custom))

# Feature Importances

In [ ]:
feature_importances = model.best_estimator_.feature_importances_
feature_importances_series = pd.Series(feature_importances, index=x_test.columns).sort_values()


In [ ]:
feature_importances_series.tail(10).plot(kind= 'barh')
plt.xlabel("Gini Importance")
plt.ylabel("Feature")
plt.title("Feature Importance");

#Export Model and Feature Medians

In [ ]:
best_rfc = RandomForestClassifier(
    max_depth = 3,
    min_samples_leaf = 90,
    min_samples_split = 325,
    n_estimators = 175,
    class_weight = 'balanced')

In [ ]:
best_rfc.fit(x_train, y_train)

In [ ]:
import joblib

# Export the trained model
joblib.dump(best_rfc, 'rf_model.pkl')

In [ ]:
import json

# Median of each of the 95 features, computed on x_train (pre-oversampling,
# pre-test-split) so it reflects the real training distribution with no
# duplicated rows (from RandomOverSampler) and no leakage from x_test.
feature_medians = x_train.median()

assert list(feature_medians.index) == list(x_train.columns)

# Save as a dict (column_name -> median). A dict/Series preserves the exact
# column order the model expects -- the Streamlit app should build its
# input row using this same order.
medians_dict = feature_medians.to_dict()
joblib.dump(medians_dict, 'feature_medians.pkl')

# Also save a JSON copy -- easy to inspect/version, and loadable without pandas
with open('feature_medians.json', 'w') as f:
    json.dump(medians_dict, f, indent=2)

print(f"Exported medians for {len(medians_dict)} features to feature_medians.pkl / .json")
feature_medians.head()